# Checking Value Labels

The purpose here is to understand whether individaul months' data can be loaded to a single relational database table or must be kept separate or transformed first before loading.
Obviously post-processing will be much easier if it can be kept to a single table.

In [21]:
import pandas as pd
import os
import json
import pymongo
from collections import Counter
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa

Retrieve value labels from MongoDB database.
Because the 2023 rebase likely differs systematically from the previous rebases, check the 2023 rebase version first.
If the differences are too substantial, it can be okay to just run analyses on 2006 onward data.
This is all for fun only anyway.

In [10]:
_mongo_details = json.load(open('.mongo.json'))
_mongo_url = 'mongodb+srv://{}:{}@{}/?retryWrites=true&w=majority'.format(*_mongo_details.values())
client = pymongo.mongo_client.MongoClient(_mongo_url, server_api=pymongo.server_api.ServerApi('1'))
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)
lfs_db = client['lfs']
lfs_var_labs = lfs_db['variable_labels']
varlabs = list(lfs_var_labs.find(filter={'file': {'$regex': r'(?i:rebase2023)'}}, projection={'_id': 0}))
lfs_val_labs = lfs_db['value_labels']
vallabs = list(lfs_val_labs.find(filter={'file': {'$regex': r'(?i:rebase2023)'}}, projection={'_id': 0}))
client.close()

Pinged your deployment. You successfully connected to MongoDB!


## Common Labelled Variables Across Years

First check whether the same variables appear in all survey years.

In [11]:
varlabs = {vl['file']: vl['labels'] for vl in varlabs}
varlabs_count = Counter(v for _, vl in varlabs.items() for v, _ in vl.items())
Counter(c for v, c in varlabs_count.items())

Counter({17: 60})

That means all 60 variables appear in all 17 syntax files with the exact same variable names.
Since that is the case, next is to check whether all value labels align.

In [12]:
vallabs = {vl['file']: {v: {i: s for s, i in l.items()} for v, l in vl['labels'].items()} for vl in vallabs}
vallabs_vars = Counter(v for _, vl in vallabs.items() for v, _ in vl.items())
vallabs = {v: {f: vl[v] for f, vl in vallabs.items()} for v in vallabs_vars.keys()}
len(vallabs.keys())

43

## Common Value Labels Across Years

There are 43 variables to check.
This is going to be somewhat lengthy.
To get around this we are going to use the following trick.
If all years' value labels agree, then treating this as an inter-rater reliability exercise will yield a perfect inter-rater reliability score.
This means that we can quickly iterate through all categorical variables and only highlight those variables with categories that do not agree and print them out for further inspection.

In [13]:
def check_value_label_agreement(data):
    assert isinstance(data, pd.DataFrame), 'Function assumes data is passed as a pandas DataFrame.'
    ratings = aggregate_raters(data)
    kappa = fleiss_kappa(ratings[0])
    return {'kappa': kappa, 'ratings': ratings[0], 'columns': ratings[1]}

Test function output on one variable.

In [14]:
check_value_label_agreement(pd.DataFrame.from_dict(vallabs[next(iter(vallabs))]).fillna('Missing'))

{'kappa': 1.0,
 'ratings': array([[ 0,  0,  0,  0, 17,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0, 17,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0, 17,  0,  0,  0,  0],
        [17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0, 17,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0, 17,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0, 17,  0,  0,  0,  0,  0,  0],
        [ 0, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 17],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 17,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0, 17,  0,  0],
        [ 0,  0, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0]]),
 'columns': array(['April', 'August', 'December', 'February', 'January', 'July',
        'June', 'March', 'May', 'November', 'October', 'September'],
       dtype=object)}

Check outputs for any variable without perfect agreement.

In [15]:
for v, vl in vallabs.items():
    rating = check_value_label_agreement(pd.DataFrame.from_dict(vl).fillna('Missing'))
    if rating['kappa'] != 1:
        print(v)
        print(rating, '\n')

LKPUBAG
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 

LKEMPLOY
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 

LKRELS
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 

LKATADS
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 

LKANSADS
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 

LKOTHERN
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 



/home/gohyc/miniconda3/envs/basic-data-science/lib/python3.10/site-packages/statsmodels/stats/inter_rater.py:267: RuntimeWarning: invalid value encountered in scalar divide
  kappa = (p_mean - p_mean_exp) / (1- p_mean_exp)
/home/gohyc/miniconda3/envs/basic-data-science/lib/python3.10/site-packages/statsmodels/stats/inter_rater.py:267: RuntimeWarning: invalid value encountered in scalar divide
  kappa = (p_mean - p_mean_exp) / (1- p_mean_exp)
/home/gohyc/miniconda3/envs/basic-data-science/lib/python3.10/site-packages/statsmodels/stats/inter_rater.py:267: RuntimeWarning: invalid value encountered in scalar divide
  kappa = (p_mean - p_mean_exp) / (1- p_mean_exp)
/home/gohyc/miniconda3/envs/basic-data-science/lib/python3.10/site-packages/statsmodels/stats/inter_rater.py:267: RuntimeWarning: invalid value encountered in scalar divide
  kappa = (p_mean - p_mean_exp) / (1- p_mean_exp)
/home/gohyc/miniconda3/envs/basic-data-science/lib/python3.10/site-packages/statsmodels/stats/inter_rater.py

## Common Variable Labels Across Years

Value labels have perfect agreement.
We can now also quickly check if there were changes in variable descriptions.

In [16]:
check_value_label_agreement(pd.DataFrame.from_dict(varlabs).fillna('Missing'))

{'kappa': 1.0,
 'ratings': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]),
 'columns': array(['Actual hours worked per week at all jobs',
        'Actual hours worked per week at main job',
        'Age in 2 and 3 year groups, 15 to 29', 'Age of youngest child',
        'Availability during the reference week',
        'Class of worker, main job', 'Current student status',
        'Duration of joblessness (months)',
        'Duration of unemployment (weeks)', 'Establishment size',
        'Firm size', 'Five-year age group of respondent',
        'Flows into unemployment',
        'Full- or part-time status at main or only job',
        'Full- or part-time status of last job',
        'Highest educational attainment',
        'Hours away from work, part-week absence only', 'Immigrant status',
        'Industry of main job', '

## Values Validation

A final check that was not done during the extraction phase is to make sure that we are retrieving the correct set of value labels.
Essentially, this means that at the very least every unique value in categorical and ordinal variables
(hopefully fully presented by the labelled variables list, but we will check manually)
must have a corresponding label.
Otherwise, this is a data quality issue that needs to be resolved
(and the MongoDB documents should be updated with the correct values and labels).

First, since all years share the same set of value labels, we can save time by examining only one set of value labels.

In [20]:
# vallabs = {v: l[next(iter(l))] for v, l in vallabs.items()}
vallabs

{'SURVMNTH': {1: 'January',
  2: 'February',
  3: 'March',
  4: 'April',
  5: 'May',
  6: 'June',
  7: 'July',
  8: 'August',
  9: 'September',
  10: 'October',
  11: 'November',
  12: 'December'},
 'LFSSTAT': {1: 'Employed, at work',
  2: 'Employed, absent from work',
  3: 'Unemployed',
  4: 'Not in labour force'},
 'PROV': {10: 'Newfoundland and Labrador',
  11: 'Prince Edward Island',
  12: 'Nova Scotia',
  13: 'New Brunswick',
  24: 'Quebec',
  35: 'Ontario',
  46: 'Manitoba',
  47: 'Saskatchewan',
  48: 'Alberta',
  59: 'British Columbia'},
 'CMA': {1: 'Québec',
  2: 'Montréal',
  3: 'Ottawa–Gatineau (Ontario part)',
  4: 'Toronto',
  5: 'Hamilton',
  6: 'Winnipeg',
  7: 'Calgary',
  8: 'Edmonton',
  9: 'Vancouver',
  0: 'Other CMA or non-CMA'},
 'AGE_12': {1: '15 to 19 years',
  2: '20 to 24 years',
  3: '25 to 29 years',
  4: '30 to 34 years',
  5: '35 to 39 years',
  6: '40 to 44 years',
  7: '45 to 49 years',
  8: '50 to 54 years',
  9: '55 to 59 years',
  10: '60 to 64 years'

Iterate through all data files to check unique values in the labelled variables.

In [65]:
data_files = [d.path for d in os.scandir('data')]
data_files = list(filter(lambda s: s.split('/')[-1].startswith('lfs'), data_files))
data_files = sorted(data_files, key=lambda s: tuple(map(int, s.split('/')[-1].rsplit('.', maxsplit=1)[0].split('-')[1:])))
data_files = list(filter(lambda s: int(s.split('/')[-1].split('-')[1])>=2006, data_files))
#   There are 208 data files.
#   If possible, don't repeat printing if the error is the same between data files.
#   Then it becomes possible to also understand how prevalent the errors are.
df_unique = dict()
df_miss = dict()
for f in data_files:
    df = pd.read_csv(f, sep='\t', encoding='utf-8')
    df = df.convert_dtypes(convert_integer=True)
    for v, vl in vallabs.items():
        df_unique_v = df[v].dropna().unique().tolist()
        df_unique_v.sort()
        if v not in df_unique or df_unique_v != df_unique[v]:
            df_unique[v] = df_unique_v[:]
        del df_unique_v
        df_miss_v = [i for i in df_unique[v] if i not in vl]
        if v not in df_miss or df_miss_v != df_miss[v]:
            df_miss[v] = df_miss_v[:]
            if len(df_miss[v]) > 0:
                print('{: >12s}:'.format(v), df_miss[v])
        del df_miss_v

      NOC_43: [41, 42, 43]


This variable is in error.
There is some additional checks not presented here but the basic story is:
- The National Occupational Classification (NOC) changed in 2021.
- The previous classification scheme had 40 level-2 occupational descriptions.
- The new scheme has 43.
- Whoever created the SPSS Syntax File forgot to change or check this.
- This can be fixed easily manually.

# Summary

All value labels and variable descriptions from 2006-2022 have perfect agreement.
The basic implication is that with a bit of clean-up where necessary, all data for 2006-2023 can be uploaded to the same data table in a relational database.

Pre-2006 variables will be examined in due time, but it is not necessary for now.
Jan 2006 to Apr 2023 is 208 months of data.